In [2]:
from dotenv import load_dotenv
from utils.agent_visualizer import print_activity, visualize_conversation

from claude_code_sdk import ClaudeCodeOptions, ClaudeSDKClient

load_dotenv()

False

# 01 - The Chief of Staff Agent

#### Introdução

In notebook 00, we built a simples research agent. In este notebook, we'll incrementally introduce key Claude Code SDK Recursos for building comprehensive agents. For each introduced recurso, we'll explain:
- **o que**: o que the recurso is
- **por que**: o que the recurso can do e por que you would want para use it
- **como**: a minimal implementation showing como para use it

se you are familiar com Claude Code, you'll notice como the SDK brings recurso parity e enables you para leverage todos of Claude Code's capabilities in a programmatic headless manner.

#### Scenario

Throughout este notebook, we'll construir an **AI Chief of Staff** for a 50-person startup aquele just raised $10M Series A. The CEO needs data-driven insights para balance aggressive growth com financial sustainability.

Our final Chief of Staff agent will:
- **Coordinate specialized subagents** for different domains
- **Aggregate insights** de multiple sources
- **Provide executive summaries** com actionable recommendations

## Basic Recursos

### recurso 0: memória com [CLAUDE.md](https://www.anthropic.com/engineering/claude-code-best-practices)

**o que**: `CLAUDE.md` arquivos serve as persistent memória e instructions for your agent. quando present in the projeto diretório, Claude Code automatically reads e incorporates este context quando you inicializar your agent.

**por que**: Instead of repeatedly providing projeto context, equipe preferências, ou standards in each interaction, you can define them once in `CLAUDE.md`. este ensures consistent behavior e reduces token Uso by avoiding redundant explanations.

**como**: 
- Have a `CLAUDE.md` arquivo in the working diretório - in our example: `chief_of_staff_agent/CLAUDE.md`
- Set the `cwd` argumento of your ClaudeSDKClient para point para diretório of your CLAUDE.md arquivo

In [ ]:
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="chief_of_staff_agent",  # Points to subdirectory with our CLAUDE.md
    )
) as agent:
    await agent.query("What's our current runway?")
    async for msg in agent.receive_response():
        if hasattr(msg, "result"):
            print(msg.result)
# The agent should know from the CLAUDE.md file: $500K burn, 20 months runway

### recurso 1: The Bash tool for Python Script Execution

**o que**: The Bash tool allows your agent para (among other things) executar Python scripts directly, enabling access para procedural knowledge, complexo computations, data analysis e other integrations aquele go beyond the agent's native capabilities.

**por que**: Our Chief of Staff might need para processo data arquivos, executar financial models ou generate visualizations based on este data. estes are todos good scenarios for using the Bash tool.

**como**: Have your Python scripts set-up in a place onde your agent can reach them e adicionar alguns context on o que they are e como they can be called. se the scripts are meant for your chief of staff agent, adicionar este context para its CLAUDE.md arquivo e se they are meant for one your subagents, adicionar said context para their MD arquivos (mais details on este mais tarde). For este Tutorial, we added five toy Exemplos para `chief_of_staff_agent/scripts`:
1. `hiring_impact.py`: Calculates como novo engineering hires affect burn rate, runway, e cash position. Essential for the `financial-analyst` subagent para model hiring scenarios against the $500K monthly burn e 20-mês runway.
2. `talent_scorer.py`: Scores candidates on technical skills, experience, culture fit, e salary expectations using weighted criteria. Core tool for the `recruiter` subagent para rank engineering candidates against TechStart's $180-220K senior engineer benchmarks.
3. `simple_calculation.py`: Performs quick financial calculations for runway, burn rate, e quarterly métricas. Utility script for chief of staff para get instant métricas sem complexo modeling.
4. `financial_forecast.py`: Models ARR growth scenarios (base/optimistic/pessimistic) given the atual $2.4M ARR growing at 15% MoM.Critical for `financial-analyst` para projeto Series B readiness e validar the $30M fundraising target.
5. `decision_matrix.py`: Creates weighted decision matrices for strategic choices like the SmartDev acquisition ou office expansion. Helps chief of staff systematically evaluate complexo decisions com multiple stakeholders e criteria.

In [ ]:
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        allowed_tools=["Bash", "Read"],
        cwd="chief_of_staff_agent",  # Points to subdirectory where our agent is defined
    )
) as agent:
    await agent.query(
        "Use your simple calculation script with a total runway of 2904829 and a monthly burn of 121938."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        if hasattr(msg, "result"):
            print("\n")
            print(msg.result)

### recurso 2: saída Styles

**o que**: saída styles allow you para use different saída styles for different audiences. Each style is defined in a markdown arquivo.

**por que**: Your agent might be used by people of different levels of expertise ou they might have different priorities. Your saída style can help differentiate entre estes segments sem having para criar a separate agent.

**como**:
- configurar a markdown arquivo per style in `chief_of_staff_agent/.claude/output-styles/`. For example, verificar out the Executive Ouput style in `.claude/output-styles/executive.md`. saída styles are defined com a simples frontmatter including two fields: name e description. Nota: Certifique-se the name in the frontmatter matches exatamente the arquivo's name (case sensitive)

> **IMPORTANT**: Output styles modify the system prompt that Claude Code has underneath, leaving out the parts focused on software engineering and giving you more control for your specific use case beyond software engineering work.

In [3]:
messages_executive = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "executive"}',
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_executive.append(msg)

messages_technical = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="chief_of_staff_agent",
        settings='{"outputStyle": "technical"}',
    )
) as agent:
    await agent.query("Tell me in two sentences about your writing output style.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages_technical.append(msg)

🤖 Thinking...
🤖 Thinking...


In [ ]:
print(messages_executive[-1].result)

In [ ]:
print(messages_technical[-1].result)

### recurso 3: Plan Mode - Strategic Planning sem Execution

**o que**: Plan mode instructs the agent para criar a detailed execution plan sem performing any actions. The agent analyzes Requisitos, proposes solutions, e outlines steps, but doesn't modificar arquivos, execute comandos, ou make changes.

**por que**: complexo tasks benefit de upfront planning para reduce erros, enable review e melhorar coordination. depois the planning phase, the agent will have a red thread para follow throughout its execution.

**como**: Just set `permission_mode="plan"`

> Note: this feature shines in Claude Code but still needs to be fully adapted for headless applications with the SDK. Namely, the agent will try calling its `ExitPlanMode()` tool, which is only relevant in the interactive mode. In this case, you can send up a follow-up query with `continue_conversation=True` for the agent to execute its plan in context.

In [9]:
messages = []
async with (
    ClaudeSDKClient(
        options=ClaudeCodeOptions(
            model="claude-opus-4-1-20250805",  # We're using Opus for this as Opus truly shines when it comes to planning!
            permission_mode="plan",
        )
    ) as agent
):
    await agent.query("Restructure our engineering team for AI focus.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Using: Glob()
✓ Tool completed
🤖 Using: Read()
✓ Tool completed
🤖 Using: Glob()
✓ Tool completed
🤖 Using: Read()
✓ Tool completed
🤖 Using: Read()
✓ Tool completed
🤖 Using: Glob()
✓ Tool completed
🤖 Thinking...
🤖 Using: ExitPlanMode()
✓ Tool completed


In [10]:
print(messages[-1].result)

As mentioned acima, the agent will parar depois creating its plan, se you want it para execute on its plan, you need para enviar a novo query com `continue_conversation=True` e removing `permission_mode="plan"` 

## avançado Recursos

### recurso 4: personalizado Slash comandos

> Note: slash commands are syntactic sugar for users, not new agent capabilities

**o que**: personalizado slash comandos are predefined prompt templates aquele usuários can trigger com shorthand syntax (e.g., `/budget-impact`). estes are **usuário-facing shortcuts**, não agent capabilities. Think of them as keyboard shortcuts aquele expand into full, well-crafted prompts.

**por que**: Your Chief of Staff will handle recurring executive questions. Instead of usuários typing complexo prompts repeatedly, they can use already vetted prompts. este improves consistência e standardization.

**como**:
- Define a markdown arquivo in `.claude/commands/`. For example, we defined one in `.claude/commands/slash-command-test.md`. Notice como the comando is defined: frontmatter com two fields (name, description) e the expanded prompt com an option para include argumentos passed on in the query.
- You can adicionar parâmetros para your prompt using `{{args}}`
- The usuário uses the slash comando in their prompt

In [11]:
# User types: "/slash-command-test this is a test"
# -> behind the scenes EXPANDS to the prompt in .claude/commands/slash-command-test.md
# In this case the expanded prompt says to simply reverse the sentence word wise

messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(model="claude-sonnet-4-20250514", cwd="chief_of_staff_agent")
) as agent:
    await agent.query("/slash-command-test this is a test")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...


In [12]:
print(messages[-1].result)

test a is this


### recurso 5: Hooks - Automated Deterministic Actions

**o que**: Hooks are Python scripts aquele you can set para execute automatically, among other events, antes (pre) ou depois (post) specific tool calls. Hooks executar **deterministically**, making them perfect for validação e audit trails.

**por que**: Imagine scenarios onde you want para Certifique-se aquele your agent has alguns guardrails (e.g., prevent dangerous operations) ou quando you want para have an audit trail. Hooks are ideal in combination com agents para allow them suficiente freedom para achieve their tarefa, while still making sure aquele the agents behave in a safe way.

**como**:
- Define hook scripts in `.claude/hooks/` -> _what_ is the behaviour aquele should be executed quando a hook is triggered
- Define hook Configuração in `.claude/settings.local.json` -> _when_ should a hook be triggered
- In este case, our hooks are configured para watch specific tool calls (WebSearch, escrever, editar, etc.)
- quando aqueles tools are called, the hook script either runs primeiro (pre tool use hook) ou depois (post tool use hook)

**Example: relatório Tracking for Compliance**

A hook para log escrever/editar operations on financial reports for audit e compliance purposes.
The hook is defined in `chief_of_staff_agent/.claude/hooks/report-tracker.py` e the logic aquele enforces it is in `chief_of_staff/.claude/settings.local.json`:


```json
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Write|Edit",
        "hooks": [
          {
            "type": "command",
            "command": "$CLAUDE_PROJECT_DIR/.claude/hooks/report-tracker.py"
          }
        ]
      }
    ]
  }
```

In [13]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        cwd="chief_of_staff_agent",
        allowed_tools=["Bash", "Write", "Edit", "MultiEdit"],
    )
) as agent:
    await agent.query(
        "Create a quick Q2 financial forecast report with our current burn rate and runway projections. Save it to our /output_reports folder."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

# The hook will track this in audit/report_history.json

🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: Bash()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: Bash()
✓ Tool completed
🤖 Thinking...
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Using: Write()
✓ Tool completed
🤖 Using: TodoWrite()
✓ Tool completed
🤖 Thinking...


se you agora navigate para `./chief_of_staff_agent/audit/report_history.json`, you will encontrar aquele it has logged aquele the agent has created e/ou made changes para your relatório. The generated relatório itself you can encontrar at `./chief_of_staff_agent/output_reports/`.

### recurso 6: Subagents via tarefa Tool

**o que**: The tarefa tool enables your agent para delegate specialized work para other subagents. estes subagents each have their own instructions, tools, e expertise.

**por que**: Adding subagents opens up a lot of possibilities:
1. Specialization: each subagent is an expert in their domain
2. Separate context: subagents have their own conversation history e tools
3. Parallellization: multiple subagents can work simultaneously on different aspects.

**como**:
- adicionar `"Task"` para allowed_tools
- Use a system prompt para instruct your agent como para delegate tasks (you can also define este its CLAUDE.md mais generally)
- criar a markdown arquivo for each agent in `.claude/agents/`. For example, verificar the one for `.claude/agents/financial-analyst.md` e notice como a (sub)agent can be defined com such an fácil e intuitive markdown arquivo: frontmatter com three fields (name, description, e tools) e its system prompt. The description is useful for the principal chief of staff agent para know quando para invoke each subagent.

In [14]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeCodeOptions(
        model="claude-sonnet-4-20250514",
        allowed_tools=["Task"],  # this enables our Chief agent to invoke subagents
        system_prompt="Delegate financial questions to the financial-analyst subagent. Do not try to answer these questions yourself.",
        cwd="chief_of_staff_agent",
    )
) as agent:
    await agent.query("Should we hire 5 engineers? Analyze the financial impact.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Thinking...
🤖 Using: Task()
🤖 Using: Bash()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
🤖 Using: Bash()
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Using: Bash()
✓ Tool completed
🤖 Using: Bash()
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [ ]:
visualize_conversation(messages)

aqui, quando our principal agent decides para use a subagent, it will:
  1. Call the tarefa tool com parâmetros like:
  ```json
    {
      "description": "Analyze hiring impact",
      "prompt": "Analyze the financial impact of hiring 5 engineers...",
      "subagent_type": "financial-analyst"
    }
  ```
  2. The tarefa tool executes the subagent in a separate context
  3. retorna results para principal Chief of Staff agent para continue processing

## Putting It todos Together

Let's agora put everything we've seen together. We will ask our agent para determine the financial impact of hiring 3 senior engineers e escrever their insights para `output_reports/hiring_decision.md`. este demonstrates todos the Recursos seen acima:
- **Bash Tool**: Used para execute the `hiring_impact.py` script para determine the impact of hiring novo engineers
- **memória**: Reads `CLAUDE.md` in diretório as context para understand the atual budgets, runway, revenue e other relevant information
- **saída style**: Different saída styles, defined in `chief_of_staff_agent/.claude/output-styles`
- **personalizado Slash comandos**: Uses the shortcut `/budget-impact` aquele expands para full prompt defined in `chief_of_staff_agent/.claude/commands`
- **Subagents**: Our `/budget_impact` comando Guias the chief of staff agent para invoke the financial-analyst subagent defined in `chief_of_staff_agent/.claude/agents` 
- **Hooks**: Hooks are defined in `chief_of_staff_agent/.claude/hooks` e configured in `chief_of_staff_agent/.claude/settings.local.json`
    - se one of our agents is updating the financial relatório, the hook should log este editar/escrever activity in the `chief_of_staff_agent/audit/report_history.json` logfile
    - se the financial analyst subagent will invoke the `hiring_impact.py` script, este will be logged in `chief_of_staff_agent/audit/tool_usage_log.json` logfile

- **Plan Mode**: se you want the chief of staff para come up com a plan for you para approve antes taking any action, uncomment the commented line abaixo

para have este ready para go, we have encapsulated the agent loop in a python arquivo, similar para o que we did in the anterior notebook. verificar out the agent.py arquivo in the `chief_of_staff_agent` subdirectory. 

todos in todos, our `send_query()` função takes in 4 parâmetros (prompt, continue_conversation, permission_mode, e output_style), everything senão is set up in the agent arquivo, namely: system prompt, max turns, allowed tools, e the working diretório.

para better visualize como este todos comes together, verificar out estes [flow e Arquitetura diagrams aquele Claude made for us :)](./chief_of_staff_agent/flow_diagram.md)


In [16]:
from chief_of_staff_agent.agent import send_query

result, messages = await send_query(
    "/budget-impact hiring 3 senior engineers. Save your insights by updating the 'hiring_decision.md' file in /output_reports or creating a new file there",
    # permission_mode="plan", # Enable this to use planning mode
    output_style="executive",
)

🤖 Thinking...
🤖 Using: Task()
🤖 Using: Bash()
✓ Tool completed
🤖 Using: Read()
🤖 Using: Read()
🤖 Using: Read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: Write()
✓ Tool completed
🤖 Thinking...


In [ ]:
visualize_conversation(messages)

## Conclusion

We've demonstrated como the Claude Code SDK enables you para construir sophisticated multi-agent systems com enterprise-grade Recursos. Starting de basic script execution com the Bash tool, we progressively introduced avançado capabilities including persistent memória com CLAUDE.md, personalizado saída styles for different audiences, strategic planning mode, slash comandos for usuário convenience, compliance hooks for guardrailing, e subagent coordination for specialized tasks.

By combining estes Recursos, we created an AI Chief of Staff capable of handling complexo executive decision-making workflows. The system delegates financial analysis para specialized subagents, maintains audit trails through hooks, adapts communication styles for different stakeholders, e provides actionable insights backed by data-driven analysis.

este foundation in avançado agentic patterns e multi-agent orchestration prepares you for building produção-ready enterprise systems. In the próximo notebook, we'll explore como para conectar our agents para externo services through Model Context Protocol (MCP) servers, dramatically expanding their capabilities beyond the built-in tools.

próximo: [02_Connecting_to_MCP_servers.ipynb](02_Connecting_to_MCP_servers.ipynb) - Learn como para extend your agents com personalizado integrations e externo data sources through MCP.